# Odd-primary operations on lens spaces

This notebook records the separate lens-space family used to validate odd-primary operations on compact face-map quotients. The construction is independent of the symmetric products of algebraic curves.

The principal examples are

- L⁷(3), detecting βP⁰: H¹ → H² and P¹: H² → H⁶;
- L¹¹(5), detecting βP⁰: H¹ → H² and P¹: H² → H¹⁰.


In [ ]:
from pathlib import Path
import sys
import time

repo = Path.cwd()
if not (repo / 'src' / 'fastop').exists():
    for candidate in (repo.parent, repo.parent.parent):
        if (candidate / 'src' / 'fastop').exists():
            repo = candidate
            break
sys.path.insert(0, str(repo / 'src'))

from fastop import __version__, spaces

print(f'fastop {__version__}')
print(f'repository: {repo}')


In [ ]:
def timed(function):
    start = time.perf_counter()
    value = function()
    return value, time.perf_counter() - start


def lens_f_vector(dimension, order):
    if dimension < 1 or dimension % 2 == 0:
        raise ValueError('dimension must be positive and odd')
    factor_count = (dimension + 1) // 2
    # In one factor: absent, q vertices, or q edges.
    cover_counts = [1]
    for _ in range(factor_count):
        next_counts = [0] * (len(cover_counts) + 2)
        for weight, coefficient in enumerate(cover_counts):
            next_counts[weight] += coefficient
            next_counts[weight + 1] += order * coefficient
            next_counts[weight + 2] += order * coefficient
        cover_counts = next_counts
    return tuple(coefficient // order for coefficient in cover_counts[1:])


## The join construction

Let q ≥ 2 and m ≥ 1. The implementation realizes the standard lens space as

**L²ᵐ⁻¹(q) = (S¹)^{*m} / C_q ≅ S²ᵐ⁻¹ / C_q.**

Each circle is represented by a q-gon. In one join factor a cell is one of:

- `None`, meaning the factor is absent;
- `(v, i)`, one of q vertices, of weight 1;
- `(e, i)`, one of q edges, of weight 2.

A label of total weight w represents a cell of dimension w − 1. For the edge e_i, the two faces are v_{i+1} and v_i, with indices read modulo q. Join faces are obtained by replacing the selected local edge or vertex by its corresponding face.

The generator of C_q sends every index i to i + 1 simultaneously in all factors. It commutes with every face map and acts freely on nonempty cell labels. `fastop` therefore takes the strict face-map quotient directly, without barycentric subdivision.


## Counting the quotient cells

The weight polynomial for one factor is `1 + qz + qz²`. Hence the cover is counted by `(1 + qz + qz²)^m`. Removing the empty label and dividing free orbits by q gives the total

**((1 + 2q)^m − 1) / q.**

The coefficient of z^{d+1}, divided by q, is the number of d-cells in the quotient.


In [ ]:
examples = [(7, 3), (11, 5), (15, 7)]
print('| model | f-vector | total cells |')
print('| --- | --- | ---: |')
for dimension, order in examples:
    f_vector = lens_f_vector(dimension, order)
    print(
        f'| L^{dimension}({order}) | `{f_vector}` | '
        f'{sum(f_vector):,} |'
    )


In [ ]:
assert lens_f_vector(7, 3) == (4, 22, 72, 153, 216, 198, 108, 27)
assert sum(lens_f_vector(7, 3)) == 800
assert lens_f_vector(11, 5) == (
    6, 81, 650, 3450, 12750, 33625, 63750, 86250,
    81250, 50625, 18750, 3125,
)
assert sum(lens_f_vector(11, 5)) == 354_312


## Cohomology and expected operations

At an odd prime p, the standard mod-p lens space in dimension 2m − 1 has

**H*(L²ᵐ⁻¹(p); F_p) ≅ Λ(a) ⊗ F_p[b]/(b^m),**

with |a| = 1, |b| = 2 and β(a) = b. The unstable identity gives P¹(b) = bᵖ. Consequently, the smallest member of this model family supporting a nonzero P¹ on b has m = p + 1 and dimension 2p + 1:

- p = 3 gives L⁷(3);
- p = 5 gives L¹¹(5).


## Standard computation: L⁷(3)

This 800-cell example is safe to execute routinely and validates both the Bockstein and reduced-power paths on dense face-map input.


In [ ]:
L7, build_seconds = timed(lambda: spaces.lens_space(7, 3))
H = L7.cohomology(p=3)
betti, cohomology_seconds = timed(
    lambda: tuple(H.betti_number(d) for d in range(8))
)
beta_rank, beta_seconds = timed(
    lambda: H.operation_rank(1, 0, bockstein=True)
)
p1_rank, p1_seconds = timed(lambda: H.operation_rank(2, 1))

assert betti == (1,) * 8
assert (beta_rank, p1_rank) == (1, 1)
print(f'cells: {sum(L7.f_vector()):,} {L7.f_vector()}')
print(f'Betti numbers: {betti}')
print(f'build: {build_seconds:.3f}s; cohomology: {cohomology_seconds:.3f}s')
print(f'beta P0 rank: {beta_rank} in {beta_seconds:.6f}s')
print(f'P1 rank: {p1_rank} in {p1_seconds:.6f}s')


## Extended computation: L¹¹(5)

The p = 5 model has 354,312 cells. The measured operation ranks are

- rank βP⁰: H¹ → H² = 1;
- rank P¹: H² → H¹⁰ = 1.

On the development machine it builds in about 14.2 seconds, evaluates P¹ in about 5.0 seconds after the relevant cohomology data is available, and peaks near 1.5 GiB. It is therefore opt-in.


In [ ]:
RUN_L11 = False

if RUN_L11:
    L11, build_seconds = timed(lambda: spaces.lens_space(11, 5))
    H = L11.cohomology(p=5)
    beta_rank, beta_seconds = timed(
        lambda: H.operation_rank(1, 0, bockstein=True)
    )
    p1_rank, p1_seconds = timed(lambda: H.operation_rank(2, 1))
    assert (beta_rank, p1_rank) == (1, 1)
    print(f'cells: {sum(L11.f_vector()):,} {L11.f_vector()}')
    print(f'build: {build_seconds:.3f}s')
    print(f'beta P0 rank: {beta_rank} in {beta_seconds:.3f}s')
    print(f'P1 rank: {p1_rank} in {p1_seconds:.3f}s')
else:
    print('Set RUN_L11 = True for the 354,312-cell p=5 validation.')


## Scaling boundary

The construction is compact relative to a triangulated quotient, but its total cell count grows as `((1 + 2p)^(p+1) − 1) / p` for the first lens space supporting P¹(b) ≠ 0. At p = 7, L¹⁵(7) already has more than 366 million cells.

This makes L⁷(3) and L¹¹(5) excellent ground-truth examples while showing that a different representation would be required to continue the same family to larger primes.
